v18: denylist and cleanup logic cleaned up

In [1]:
from pathlib import Path
import json
import sys
import importlib

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from pestclef.config import ExperimentConfig
import pestclef.modernbert as mb
import pestclef.pipeline as ppl

importlib.reload(mb)
importlib.reload(ppl)

BASE_ARTIFACTS = ROOT / "artifacts" / "modernbert_e2e_v14"
RUN_ARTIFACTS = ROOT / "artifacts" / "modernbert_e2e_v18"

base_payload = json.loads((BASE_ARTIFACTS / "experiment_config.json").read_text(encoding="utf-8"))["config"]
for key in ["project_root", "data_dir", "json_dir", "docs_dir", "artifacts_dir"]:
    base_payload[key] = Path(base_payload[key])

# v18: relax cleanup denylist for clearly valid gold mentions
for value in [
    "to date",
    "so far",
    "for the first time",
    "since then",
    "2022",
]:
    mb.TYPE_DENYLISTS["Date"].discard(value)

for value in [
    "asian countries",
    "piedmont",
]:
    mb.TYPE_DENYLISTS["Location"].discard(value)

config = ExperimentConfig(**base_payload)
config.artifacts_dir = RUN_ARTIFACTS

result = ppl.run_dev_evaluation(config)

metrics = json.loads((RUN_ARTIFACTS / "dev_end_to_end_metrics.json").read_text(encoding="utf-8"))
mention = json.loads((RUN_ARTIFACTS / "dev_mention_metrics.json").read_text(encoding="utf-8"))
entity = json.loads((RUN_ARTIFACTS / "dev_entity_metrics.json").read_text(encoding="utf-8"))
cand = json.loads((RUN_ARTIFACTS / "dev_candidate_recall.json").read_text(encoding="utf-8"))
err = json.loads((RUN_ARTIFACTS / "dev_end_to_end_error_report.json").read_text(encoding="utf-8"))

summary = {
    "run": "v18",
    "artifacts_dir": str(RUN_ARTIFACTS),
    "micro_f1": metrics["micro"]["f1"],
    "macro_f1": metrics["macro"]["f1"],
    "mention_micro_f1": mention["micro"]["f1"],
    "entity_micro_f1": entity["micro"]["f1"],
    "candidate_recall": cand["summary"]["candidate_recall"],
    "prediction_recall_after_candidates": cand["summary"]["prediction_recall_after_candidates"],
    "bad_span": cand["summary"]["bad_span"],
    "candidate_missed_by_pruning": cand["summary"]["candidate_missed_by_pruning"],
    "bad_entity_fp": err["failure_buckets"].get("bad_entity_fp", 0),
    "classifier_fp_on_valid_candidate": err["failure_buckets"].get("classifier_fp_on_valid_candidate", 0),
}
print(json.dumps(summary, indent=2))


/Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/.venv/lib/python3.14/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForTokenClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "run": "v18",
  "artifacts_dir": "/Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/artifacts/modernbert_e2e_v18",
  "micro_f1": 0.23407643312101914,
  "macro_f1": 0.20304688760312264,
  "mention_micro_f1": 0.5489276139410187,
  "entity_micro_f1": 0.36877523553162855,
  "candidate_recall": 0.2961832061068702,
  "prediction_recall_after_candidates": 0.8092783505154639,
  "bad_span": 408,
  "candidate_missed_by_pruning": 33,
  "bad_entity_fp": 229,
  "classifier_fp_on_valid_candidate": 193
}


v19:

In [2]:
from pathlib import Path
import json
import sys
import importlib
import re

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from pestclef.config import ExperimentConfig
import pestclef.modernbert as mb
import pestclef.pipeline as ppl

importlib.reload(mb)
importlib.reload(ppl)

BASE_ARTIFACTS = ROOT / "artifacts" / "modernbert_e2e_v14"
RUN_ARTIFACTS = ROOT / "artifacts" / "modernbert_e2e_v19"

base_payload = json.loads((BASE_ARTIFACTS / "experiment_config.json").read_text(encoding="utf-8"))["config"]
for key in ["project_root", "data_dir", "json_dir", "docs_dir", "artifacts_dir"]:
    base_payload[key] = Path(base_payload[key])

# v18 relaxations carried forward
for value in [
    "to date",
    "so far",
    "for the first time",
    "since then",
    "2022",
]:
    mb.TYPE_DENYLISTS["Date"].discard(value)

for value in [
    "asian countries",
    "piedmont",
]:
    mb.TYPE_DENYLISTS["Location"].discard(value)

# v19: add targeted junk / boilerplate filter
orig_cleanup = mb._passes_type_specific_cleanup

NON_DATE_STOPWORDS = {
    "the", "we", "it", "this", "that", "these", "those", "a", "an"
}
JUNK_EXACT = {
    "genbank", "supplementary", "fig", "figure", "table", "tables", "cv",
    "https", "http", "www", "doi", "doi.org"
}
JUNK_SUBSTRINGS = (
    "https://", "http://", "www.", "doi.org/", "dx.doi.org/"
)

def extra_junk_filter(surface: str, entity_type: str) -> bool:
    normalized = mb.normalize_text(surface)
    if not normalized:
        return False

    if any(token in normalized for token in JUNK_SUBSTRINGS):
        return False
    if normalized in JUNK_EXACT:
        return False

    if entity_type != "Date" and normalized in NON_DATE_STOPWORDS:
        return False

    if entity_type != "Date" and re.fullmatch(r"[a-z]\.?", normalized):
        return False

    if entity_type != "Date" and "/" in normalized and any(ch.isdigit() for ch in normalized):
        return False

    if entity_type != "Date" and re.fullmatch(r"(?:\d+[./-])+\d+", normalized):
        return False

    # obvious bibliographic / header fragments
    if entity_type != "Date" and re.fullmatch(r"(fig|figure|table)\s*\d+[a-z]?", normalized):
        return False

    return True

def patched_cleanup(surface: str, entity_type: str, config):
    if not orig_cleanup(surface, entity_type, config):
        return False
    return extra_junk_filter(surface, entity_type)

mb._passes_type_specific_cleanup = patched_cleanup

config = ExperimentConfig(**base_payload)
config.artifacts_dir = RUN_ARTIFACTS

result = ppl.run_dev_evaluation(config)

metrics = json.loads((RUN_ARTIFACTS / "dev_end_to_end_metrics.json").read_text(encoding="utf-8"))
mention = json.loads((RUN_ARTIFACTS / "dev_mention_metrics.json").read_text(encoding="utf-8"))
entity = json.loads((RUN_ARTIFACTS / "dev_entity_metrics.json").read_text(encoding="utf-8"))
cand = json.loads((RUN_ARTIFACTS / "dev_candidate_recall.json").read_text(encoding="utf-8"))
err = json.loads((RUN_ARTIFACTS / "dev_end_to_end_error_report.json").read_text(encoding="utf-8"))

summary = {
    "run": "v19",
    "artifacts_dir": str(RUN_ARTIFACTS),
    "micro_f1": metrics["micro"]["f1"],
    "macro_f1": metrics["macro"]["f1"],
    "mention_micro_f1": mention["micro"]["f1"],
    "entity_micro_f1": entity["micro"]["f1"],
    "candidate_recall": cand["summary"]["candidate_recall"],
    "prediction_recall_after_candidates": cand["summary"]["prediction_recall_after_candidates"],
    "bad_span": cand["summary"]["bad_span"],
    "candidate_missed_by_pruning": cand["summary"]["candidate_missed_by_pruning"],
    "bad_entity_fp": err["failure_buckets"].get("bad_entity_fp", 0),
    "classifier_fp_on_valid_candidate": err["failure_buckets"].get("classifier_fp_on_valid_candidate", 0),
}
print(json.dumps(summary, indent=2))


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForTokenClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "run": "v19",
  "artifacts_dir": "/Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/artifacts/modernbert_e2e_v19",
  "micro_f1": 0.18552875695732837,
  "macro_f1": 0.16964391431509565,
  "mention_micro_f1": 0.5354645354645354,
  "entity_micro_f1": 0.3614564831261101,
  "candidate_recall": 0.2916030534351145,
  "prediction_recall_after_candidates": 0.5602094240837696,
  "bad_span": 412,
  "candidate_missed_by_pruning": 32,
  "bad_entity_fp": 176,
  "classifier_fp_on_valid_candidate": 128
}


v20:

In [3]:
from pathlib import Path
import json
import sys
import importlib
import re

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from pestclef.config import ExperimentConfig
import pestclef.modernbert as mb
import pestclef.pipeline as ppl

importlib.reload(mb)
importlib.reload(ppl)

BASE_ARTIFACTS = ROOT / "artifacts" / "modernbert_e2e_v14"
RUN_ARTIFACTS = ROOT / "artifacts" / "modernbert_e2e_v20"

base_payload = json.loads((BASE_ARTIFACTS / "experiment_config.json").read_text(encoding="utf-8"))["config"]
for key in ["project_root", "data_dir", "json_dir", "docs_dir", "artifacts_dir"]:
    base_payload[key] = Path(base_payload[key])

# v18 relaxations carried forward
for value in [
    "to date",
    "so far",
    "for the first time",
    "since then",
    "2022",
]:
    mb.TYPE_DENYLISTS["Date"].discard(value)

for value in [
    "asian countries",
    "piedmont",
]:
    mb.TYPE_DENYLISTS["Location"].discard(value)

# v19 junk filter carried forward
orig_cleanup = mb._passes_type_specific_cleanup

NON_DATE_STOPWORDS = {
    "the", "we", "it", "this", "that", "these", "those", "a", "an"
}
JUNK_EXACT = {
    "genbank", "supplementary", "fig", "figure", "table", "tables", "cv",
    "https", "http", "www", "doi", "doi.org"
}
JUNK_SUBSTRINGS = (
    "https://", "http://", "www.", "doi.org/", "dx.doi.org/"
)

def extra_junk_filter(surface: str, entity_type: str) -> bool:
    normalized = mb.normalize_text(surface)
    if not normalized:
        return False
    if any(token in normalized for token in JUNK_SUBSTRINGS):
        return False
    if normalized in JUNK_EXACT:
        return False
    if entity_type != "Date" and normalized in NON_DATE_STOPWORDS:
        return False
    if entity_type != "Date" and re.fullmatch(r"[a-z]\.?", normalized):
        return False
    if entity_type != "Date" and "/" in normalized and any(ch.isdigit() for ch in normalized):
        return False
    if entity_type != "Date" and re.fullmatch(r"(?:\d+[./-])+\d+", normalized):
        return False
    if entity_type != "Date" and re.fullmatch(r"(fig|figure|table)\s*\d+[a-z]?", normalized):
        return False
    return True

def patched_cleanup(surface: str, entity_type: str, config):
    if not orig_cleanup(surface, entity_type, config):
        return False
    return extra_junk_filter(surface, entity_type)

mb._passes_type_specific_cleanup = patched_cleanup

# v20: inject high-value alias rescue spans into the hybrid span stage
orig_build_hybrid = mb.build_hybrid_span_candidates

SEED_ALIASES = {
    "Pest": [
        "B. xylophilus",
        "Bursaphelenchus xylophilus",
        "X. fastidiosa",
        "Xylella fastidiosa",
        "TR4",
        "FocTR4",
        "Japanese beetle",
        "Spodoptera frugiperda",
    ],
    "Vector": [
        "African Psila",
        "Philaenus spumarius",
        "P. spumarius",
        "Diaphorina citri",
        "Diaphorina citri. Kuwayama",
    ],
    "Disease": [
        "Panama disease",
        "pine wilt disease",
        "HLB",
        "Huanglongbing",
        "greening",
    ],
    "Plant": [
        "citrus",
        "citrus fruits",
        "banana",
        "bananas",
        "Cavendish banana",
        "olive trees",
    ],
}

def make_seed_spans(document, mention_detector):
    text_cf = document.text.casefold()
    seeded = []
    seen = set()

    for entity_type, aliases in SEED_ALIASES.items():
        threshold = float(mention_detector.mention_thresholds.get(entity_type, 0.0))
        confidence = max(
            0.80,
            float(mention_detector.config.mention_hybrid_lexicon_confidence),
            threshold + 0.08,
        )

        for alias in aliases:
            pattern = re.compile(rf"(?<!\w){re.escape(alias.casefold())}(?!\w)")
            for match in pattern.finditer(text_cf):
                key = (entity_type, match.start(), match.end())
                if key in seen:
                    continue
                seen.add(key)
                seeded.append(
                    mb.PredictedMentionSpan(
                        entity_type=entity_type,
                        start=match.start(),
                        end=match.end(),
                        confidence=confidence,
                    )
                )
    return seeded

def patched_build_hybrid(document, mention_detector, lexicon):
    neural_spans, lexicon_spans, blended_spans = orig_build_hybrid(document, mention_detector, lexicon)
    seeded_spans = make_seed_spans(document, mention_detector)
    blended_spans = mb.combine_span_candidates([blended_spans, seeded_spans])
    lexicon_spans = mb.combine_span_candidates([lexicon_spans, seeded_spans])
    return neural_spans, lexicon_spans, blended_spans

mb.build_hybrid_span_candidates = patched_build_hybrid

config = ExperimentConfig(**base_payload)
config.artifacts_dir = RUN_ARTIFACTS
config.mention_hybrid_lexicon_confidence = 0.60
config.mention_threshold_candidate_recall_weight = 0.85
config.mention_threshold_mention_f1_weight = 0.15

result = ppl.run_dev_evaluation(config)

metrics = json.loads((RUN_ARTIFACTS / "dev_end_to_end_metrics.json").read_text(encoding="utf-8"))
mention = json.loads((RUN_ARTIFACTS / "dev_mention_metrics.json").read_text(encoding="utf-8"))
entity = json.loads((RUN_ARTIFACTS / "dev_entity_metrics.json").read_text(encoding="utf-8"))
cand = json.loads((RUN_ARTIFACTS / "dev_candidate_recall.json").read_text(encoding="utf-8"))
err = json.loads((RUN_ARTIFACTS / "dev_end_to_end_error_report.json").read_text(encoding="utf-8"))

summary = {
    "run": "v20",
    "artifacts_dir": str(RUN_ARTIFACTS),
    "micro_f1": metrics["micro"]["f1"],
    "macro_f1": metrics["macro"]["f1"],
    "mention_micro_f1": mention["micro"]["f1"],
    "entity_micro_f1": entity["micro"]["f1"],
    "vector_mention_recall": mention["per_type"]["Vector"]["recall"],
    "date_mention_recall": mention["per_type"]["Date"]["recall"],
    "candidate_recall": cand["summary"]["candidate_recall"],
    "prediction_recall_after_candidates": cand["summary"]["prediction_recall_after_candidates"],
    "bad_span": cand["summary"]["bad_span"],
    "candidate_missed_by_pruning": cand["summary"]["candidate_missed_by_pruning"],
    "bad_entity_fp": err["failure_buckets"].get("bad_entity_fp", 0),
    "classifier_fp_on_valid_candidate": err["failure_buckets"].get("classifier_fp_on_valid_candidate", 0),
}
print(json.dumps(summary, indent=2))


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForTokenClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "run": "v20",
  "artifacts_dir": "/Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/artifacts/modernbert_e2e_v20",
  "micro_f1": 0.18706293706293706,
  "macro_f1": 0.16523965975430935,
  "mention_micro_f1": 0.5457537853851218,
  "entity_micro_f1": 0.35431654676258995,
  "vector_mention_recall": 0.625,
  "date_mention_recall": 0.2,
  "candidate_recall": 0.30229007633587784,
  "prediction_recall_after_candidates": 0.5909090909090909,
  "bad_span": 408,
  "candidate_missed_by_pruning": 28,
  "bad_entity_fp": 209,
  "classifier_fp_on_valid_candidate": 142
}


Comparison v14, v18, v19, v20:

In [4]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()

RUNS = {
    "v14": ROOT / "artifacts" / "modernbert_e2e_v14",
    "v18": ROOT / "artifacts" / "modernbert_e2e_v18",
    "v19": ROOT / "artifacts" / "modernbert_e2e_v19",
    "v20": ROOT / "artifacts" / "modernbert_e2e_v20",
}

rows = []
for name, run_dir in RUNS.items():
    if not run_dir.exists():
        rows.append({"run": name, "status": "missing"})
        continue

    try:
        metrics = json.loads((run_dir / "dev_end_to_end_metrics.json").read_text(encoding="utf-8"))
        mention = json.loads((run_dir / "dev_mention_metrics.json").read_text(encoding="utf-8"))
        entity = json.loads((run_dir / "dev_entity_metrics.json").read_text(encoding="utf-8"))
        cand = json.loads((run_dir / "dev_candidate_recall.json").read_text(encoding="utf-8"))
        err = json.loads((run_dir / "dev_end_to_end_error_report.json").read_text(encoding="utf-8"))
        pair = json.loads((run_dir / "dev_pair_volume.json").read_text(encoding="utf-8"))

        rows.append({
            "run": name,
            "status": "ok",
            "micro_f1": metrics["micro"]["f1"],
            "macro_f1": metrics["macro"]["f1"],
            "mention_micro_f1": mention["micro"]["f1"],
            "entity_micro_f1": entity["micro"]["f1"],
            "date_recall": mention["per_type"]["Date"]["recall"],
            "location_recall": mention["per_type"]["Location"]["recall"],
            "vector_recall": mention["per_type"]["Vector"]["recall"],
            "candidate_recall": cand["summary"]["candidate_recall"],
            "pred_recall_after_candidates": cand["summary"]["prediction_recall_after_candidates"],
            "bad_span": cand["summary"]["bad_span"],
            "candidate_pruning_miss": cand["summary"]["candidate_missed_by_pruning"],
            "valid_candidate": cand["summary"]["valid_candidate"],
            "predicted_gold_candidates": cand["summary"]["predicted"],
            "bad_entity_fp": err["failure_buckets"].get("bad_entity_fp", 0),
            "classifier_fp_valid": err["failure_buckets"].get("classifier_fp_on_valid_candidate", 0),
            "classifier_fn_valid": err["failure_buckets"].get("classifier_fn_on_valid_candidate", 0),
            "pair_total": pair["summary"]["candidate_pairs"]["total"],
            "final_entities": pair["summary"]["final_entities"]["total"],
            "final_mentions": pair["summary"]["final_mentions"]["total"],
        })
    except FileNotFoundError as e:
        rows.append({"run": name, "status": f"missing_file: {e.filename}"})

df = pd.DataFrame(rows)

display_cols = [
    "run",
    "status",
    "micro_f1",
    "macro_f1",
    "mention_micro_f1",
    "entity_micro_f1",
    "date_recall",
    "location_recall",
    "vector_recall",
    "candidate_recall",
    "pred_recall_after_candidates",
    "bad_span",
    "candidate_pruning_miss",
    "bad_entity_fp",
    "classifier_fp_valid",
    "classifier_fn_valid",
    "pair_total",
    "final_entities",
    "final_mentions",
]

existing_cols = [c for c in display_cols if c in df.columns]
df = df[existing_cols].sort_values(by=["status", "micro_f1"], ascending=[True, False], na_position="last")

def highlight_best(s):
    if s.name in {"run", "status"}:
        return [""] * len(s)
    numeric = pd.to_numeric(s, errors="coerce")
    if numeric.isna().all():
        return [""] * len(s)

    if s.name in {
        "bad_span",
        "candidate_pruning_miss",
        "bad_entity_fp",
        "classifier_fp_valid",
        "classifier_fn_valid",
    }:
        best = numeric.min()
    else:
        best = numeric.max()
    return ["background-color: #d9f2d9" if pd.notna(v) and v == best else "" for v in numeric]

styled = (
    df.style
    .format({
        "micro_f1": "{:.4f}",
        "macro_f1": "{:.4f}",
        "mention_micro_f1": "{:.4f}",
        "entity_micro_f1": "{:.4f}",
        "date_recall": "{:.4f}",
        "location_recall": "{:.4f}",
        "vector_recall": "{:.4f}",
        "candidate_recall": "{:.4f}",
        "pred_recall_after_candidates": "{:.4f}",
    })
    .apply(highlight_best, axis=0)
)

display(styled)

if "micro_f1" in df.columns:
    ok_df = df[df["status"] == "ok"].copy()
    if not ok_df.empty:
        best_row = ok_df.sort_values("micro_f1", ascending=False).iloc[0]
        print("Best run by micro F1:")
        print(best_row.to_dict())


,run,status,micro_f1,macro_f1,mention_micro_f1,entity_micro_f1,date_recall,location_recall,vector_recall,candidate_recall,pred_recall_after_candidates,bad_span,candidate_pruning_miss,bad_entity_fp,classifier_fp_valid,classifier_fn_valid,pair_total,final_entities,final_mentions
1,v18,ok,0.2341,0.2030,0.5489,0.3688,0.2000,0.4845,0.0312,0.2962,0.8093,408,33,229,193,57,1758,710,1375
0,v14,ok,0.2263,0.1844,0.5343,0.3497,0.1395,0.4577,0.0312,0.2794,0.7322,427,29,191,139,62,1642,706,1333
3,v20,ok,0.1871,0.1652,0.5458,0.3543,0.2000,0.4845,0.6250,0.3023,0.5909,408,28,209,142,99,1672,705,1429
2,v19,ok,0.1855,0.1696,0.5355,0.3615,0.2000,0.4784,0.0312,0.2916,0.5602,412,32,176,128,97,1769,733,1394


Best run by micro F1:
{'run': 'v18', 'status': 'ok', 'micro_f1': 0.23407643312101914, 'macro_f1': 0.20304688760312264, 'mention_micro_f1': 0.5489276139410187, 'entity_micro_f1': 0.36877523553162855, 'date_recall': 0.2, 'location_recall': 0.4845360824742268, 'vector_recall': 0.03125, 'candidate_recall': 0.2961832061068702, 'pred_recall_after_candidates': 0.8092783505154639, 'bad_span': 408, 'candidate_pruning_miss': 33, 'bad_entity_fp': 229, 'classifier_fp_valid': 193, 'classifier_fn_valid': 57, 'pair_total': 1758, 'final_entities': 710, 'final_mentions': 1375}


Common Setup for ensembling predictions

In [5]:
from pathlib import Path
import csv
import json
import re
import sys
import importlib

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from transformers import AutoTokenizer, AutoConfig, AutoModelForTokenClassification

from pestclef.config import ExperimentConfig
from pestclef.data import load_documents, deduplicate_edges
from pestclef.features import RelationSchema
from pestclef.mention_detection import MentionLexicon
from pestclef.schema import RelationEdge
from pestclef.submission import validate_submission_rows
import pestclef.modernbert as mb
import pestclef.pipeline as ppl

importlib.reload(mb)
importlib.reload(ppl)


<module 'pestclef.pipeline' from '/Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/src/pestclef/pipeline.py'>

Helpers To Load Saved Models And Write Submissions

In [6]:
def load_experiment_config(artifacts_dir: Path) -> ExperimentConfig:
    payload = json.loads((artifacts_dir / "experiment_config.json").read_text(encoding="utf-8"))["config"]
    for key in ["project_root", "data_dir", "json_dir", "docs_dir", "artifacts_dir"]:
        payload[key] = Path(payload[key])
    return ExperimentConfig(**payload)

def load_saved_mention_detector(artifacts_dir: Path, config: ExperimentConfig) -> mb.ModernBertMentionDetector:
    model_dir = artifacts_dir / "mention_model"
    metadata = json.loads((model_dir / "metadata.json").read_text(encoding="utf-8"))
    labels = metadata["labels"]
    label_to_id = {label: i for i, label in enumerate(labels)}
    id_to_label = {i: label for i, label in enumerate(labels)}

    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model_config = AutoConfig.from_pretrained(model_dir)
    model = AutoModelForTokenClassification.from_pretrained(model_dir, config=model_config)

    return mb.ModernBertMentionDetector(
        tokenizer=tokenizer,
        model=model,
        label_to_id=label_to_id,
        id_to_label=id_to_label,
        device=mb.resolve_torch_device(config.device),
        config=config,
        mention_thresholds={k: float(v) for k, v in metadata["mention_thresholds"].items()},
    )

def load_saved_relation_model(artifacts_dir: Path, config: ExperimentConfig) -> mb.ModernBertRelationModel:
    return mb.ModernBertRelationModel.load(artifacts_dir / "relation_model", config)

def write_submission_csv(predictions_by_doc: dict, output_path: Path) -> Path:
    rows = []
    for doc_id in sorted(predictions_by_doc):
        edges = predictions_by_doc[doc_id]
        payload = [
            {"subject": e.subject, "predicate": e.predicate, "object": e.object}
            for e in edges
        ]
        rows.append({
            "doc_id": doc_id,
            "knowledge_graph": json.dumps(payload, ensure_ascii=False),
        })

    errors = validate_submission_rows(rows)
    if errors:
        raise ValueError(f"Submission validation failed: {errors[:10]}")

    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["doc_id", "knowledge_graph"])
        writer.writeheader()
        writer.writerows(rows)

    return output_path

def predict_test_submission_from_saved_artifacts(
    artifacts_dir: Path,
    output_path: Path,
) -> tuple[Path, dict]:
    config = load_experiment_config(artifacts_dir)
    config.project_root = ROOT
    config.artifacts_dir = artifacts_dir

    test_documents = load_documents("test", config)
    train_documents = load_documents("train", config)

    schema = RelationSchema.from_documents(train_documents)
    mention_detector = load_saved_mention_detector(artifacts_dir, config)
    relation_model = load_saved_relation_model(artifacts_dir, config)
    mention_lexicon = MentionLexicon.from_documents(train_documents, config) if config.mention_hybrid_lexicon else None

    predictions = {}
    for document in test_documents:
        _, _, _, _, predicted_entities_by_doc, _ = ppl._collect_predicted_entity_state(
            [document], mention_detector, mention_lexicon
        )
        predicted_entities = predicted_entities_by_doc[document.doc_id]
        examples, entity_pairs = mb.build_relation_inference_examples(
            document,
            predicted_entities,
            schema,
            config,
        )
        edges = []
        for predicted_labels, (subject, obj) in zip(relation_model.predict_labels(examples), entity_pairs):
            for label in predicted_labels:
                if schema.is_valid_pair(label, subject.entity_type, obj.entity_type):
                    edges.append(
                        RelationEdge(
                            subject=subject.canonical_form,
                            predicate=label,
                            object=obj.canonical_form,
                        )
                    )
        predictions[document.doc_id] = deduplicate_edges(edges)

    write_submission_csv(predictions, output_path)
    return output_path, predictions


Apply The v18 Runtime Patch And Generate The Best Single Submission



In [7]:
# Reapply the v18 notebook-time cleanup relaxation for inference.
importlib.reload(mb)
importlib.reload(ppl)

for value in [
    "to date",
    "so far",
    "for the first time",
    "since then",
    "2022",
]:
    mb.TYPE_DENYLISTS["Date"].discard(value)

for value in [
    "asian countries",
    "piedmont",
]:
    mb.TYPE_DENYLISTS["Location"].discard(value)

V18_ARTIFACTS = ROOT / "artifacts" / "modernbert_e2e_v18"
V18_SUBMISSION = ROOT / "submission_modernbert_e2e_v18_best_single.csv"

single_path, v18_predictions = predict_test_submission_from_saved_artifacts(
    V18_ARTIFACTS,
    V18_SUBMISSION,
)

print(f"Best single submission written to: {single_path}")
print(f"Total predicted edges: {sum(len(v) for v in v18_predictions.values())}")


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Best single submission written to: /Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/submission_modernbert_e2e_v18_best_single.csv
Total predicted edges: 1227


Generate The v14 Submission From Saved Artifacts

In [8]:
importlib.reload(mb)
importlib.reload(ppl)

V14_ARTIFACTS = ROOT / "artifacts" / "modernbert_e2e_v14"
V14_SUBMISSION = ROOT / "submission_modernbert_e2e_v14_for_ensemble.csv"

v14_path, v14_predictions = predict_test_submission_from_saved_artifacts(
    V14_ARTIFACTS,
    V14_SUBMISSION,
)

print(f"v14 ensemble base submission written to: {v14_path}")
print(f"Total predicted edges: {sum(len(v) for v in v14_predictions.values())}")


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

v14 ensemble base submission written to: /Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/submission_modernbert_e2e_v14_for_ensemble.csv
Total predicted edges: 947


Build The Best Ensemble (v14 ∩ v18)



In [9]:
def edge_key(edge: RelationEdge):
    return (edge.subject, edge.predicate, edge.object)

def intersect_predictions(pred_a: dict, pred_b: dict) -> dict:
    out = {}
    for doc_id in sorted(set(pred_a) | set(pred_b)):
        a = {edge_key(e): e for e in pred_a.get(doc_id, [])}
        b = {edge_key(e): e for e in pred_b.get(doc_id, [])}
        shared = sorted(set(a) & set(b))
        out[doc_id] = [a[k] for k in shared]
    return out

ENSEMBLE_SUBMISSION = ROOT / "submission_ensemble_v14_v18_intersection.csv"

ensemble_predictions = intersect_predictions(v14_predictions, v18_predictions)
write_submission_csv(ensemble_predictions, ENSEMBLE_SUBMISSION)

print(f"Best ensemble submission written to: {ENSEMBLE_SUBMISSION}")
print(f"Total predicted edges: {sum(len(v) for v in ensemble_predictions.values())}")


Best ensemble submission written to: /Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/submission_ensemble_v14_v18_intersection.csv
Total predicted edges: 606


Quick summary

In [10]:
summary = {
    "best_single_submission": str(V18_SUBMISSION),
    "best_single_edges": sum(len(v) for v in v18_predictions.values()),
    "best_ensemble_submission": str(ENSEMBLE_SUBMISSION),
    "best_ensemble_edges": sum(len(v) for v in ensemble_predictions.values()),
    "dev_reference": {
        "v18_micro_f1": 0.23407643312101914,
        "v14_v18_intersection_micro_f1": 0.241919,
    },
}
print(json.dumps(summary, indent=2))


{
  "best_single_submission": "/Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/submission_modernbert_e2e_v18_best_single.csv",
  "best_single_edges": 1227,
  "best_ensemble_submission": "/Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/submission_ensemble_v14_v18_intersection.csv",
  "best_ensemble_edges": 606,
  "dev_reference": {
    "v18_micro_f1": 0.23407643312101914,
    "v14_v18_intersection_micro_f1": 0.241919
  }
}


Backup ensembles - heuristic v9+v14 and heuristic v9+v18 ensembling

In [11]:
# Backup ensembles:
# 1. v9 + v14 heuristic
# 2. v9 + v18 heuristic
#
# Assumes you already ran the earlier cells and have:
# - ROOT
# - predict_test_submission_from_saved_artifacts(...)
# - write_submission_csv(...)
# - V14_ARTIFACTS
# - V18_ARTIFACTS
# - v14_predictions
# - v18_predictions

import importlib

# Load v9 predictions from its saved artifacts if not already present
if "v9_predictions" not in globals():
    importlib.reload(mb)
    importlib.reload(ppl)

    V9_ARTIFACTS = ROOT / "artifacts" / "modernbert_e2e_v9"
    V9_SUBMISSION = ROOT / "submission_modernbert_e2e_v9_for_ensemble.csv"

    _, v9_predictions = predict_test_submission_from_saved_artifacts(
        V9_ARTIFACTS,
        V9_SUBMISSION,
    )

def heuristic_ensemble_predictions(pred_a: dict, pred_b: dict, prefer_b_for=("Affects", "Located_in"), prefer_a_for=("Found_on", "Occurs_on")) -> dict:
    out = {}
    for doc_id in sorted(set(pred_a) | set(pred_b)):
        a = {edge_key(e): e for e in pred_a.get(doc_id, [])}
        b = {edge_key(e): e for e in pred_b.get(doc_id, [])}

        merged_keys = set()
        all_keys = set(a) | set(b)

        for k in all_keys:
            _, pred, _ = k
            in_a = k in a
            in_b = k in b

            if in_a and in_b:
                merged_keys.add(k)
            elif pred in prefer_b_for:
                if in_b:
                    merged_keys.add(k)
            elif pred in prefer_a_for:
                if in_a:
                    merged_keys.add(k)
            elif pred in ("Causes", "Transmits", "Dispersed_by"):
                if in_a or in_b:
                    merged_keys.add(k)

        out[doc_id] = [b.get(k) or a[k] for k in sorted(merged_keys)]
    return out

# v9 + v14 heuristic
ensemble_v9_v14_predictions = heuristic_ensemble_predictions(
    v9_predictions,
    v14_predictions,
    prefer_b_for=("Affects", "Located_in"),
    prefer_a_for=("Found_on", "Occurs_on"),
)
ENSEMBLE_V9_V14_PATH = ROOT / "submission_ensemble_v9_v14_heuristic.csv"
write_submission_csv(ensemble_v9_v14_predictions, ENSEMBLE_V9_V14_PATH)

# v9 + v18 heuristic
ensemble_v9_v18_predictions = heuristic_ensemble_predictions(
    v9_predictions,
    v18_predictions,
    prefer_b_for=("Affects", "Located_in"),
    prefer_a_for=("Found_on", "Occurs_on"),
)
ENSEMBLE_V9_V18_PATH = ROOT / "submission_ensemble_v9_v18_heuristic.csv"
write_submission_csv(ensemble_v9_v18_predictions, ENSEMBLE_V9_V18_PATH)

print(json.dumps({
    "v9_edges": sum(len(v) for v in v9_predictions.values()),
    "v14_edges": sum(len(v) for v in v14_predictions.values()),
    "v18_edges": sum(len(v) for v in v18_predictions.values()),
    "submission_ensemble_v9_v14_heuristic": str(ENSEMBLE_V9_V14_PATH),
    "submission_ensemble_v9_v14_edges": sum(len(v) for v in ensemble_v9_v14_predictions.values()),
    "submission_ensemble_v9_v18_heuristic": str(ENSEMBLE_V9_V18_PATH),
    "submission_ensemble_v9_v18_edges": sum(len(v) for v in ensemble_v9_v18_predictions.values()),
}, indent=2))


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

{
  "v9_edges": 891,
  "v14_edges": 947,
  "v18_edges": 1227,
  "submission_ensemble_v9_v14_heuristic": "/Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/submission_ensemble_v9_v14_heuristic.csv",
  "submission_ensemble_v9_v14_edges": 856,
  "submission_ensemble_v9_v18_heuristic": "/Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/submission_ensemble_v9_v18_heuristic.csv",
  "submission_ensemble_v9_v18_edges": 1067
}
